# Preparing the Dataset for BERTopic

In [1]:
# Import libraries
import os
import pandas as pd
# Import dataset that was created in Step 1
#df = pd.read_csv("dpdr_preprocessed.csv")
df = pd.read_csv("dpdr_preprocessed_final.csv")


len(df)

922

In [3]:
# Which Python Version am I using?
import sys

print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
#Which version is being used?
import bertopic
print("BERTopic Version:",bertopic.__version__)


BERTopic Version: 0.17.4


In [4]:
# Create a list for BERTopic
docs = df["document"].tolist()
# Did it work?
docs[0]

"Am I too inteligent for life or am i just coping? 've constructed a fake world in my mind, the real world became a second priority. My fake world is unexplainable to others, but it offers me comfort while the real world seems like hell, even though my life isn't especially traumatic, I'm just extremely lonely, with not a single soul to talk to besides my therapist of one month (this experience of somebody listending to me has been extremely validating, and I'm able to understand myself a little bit more).I've been sufferering from a derealization so bad that I didn't even think people were real, even my parents whom I avoided. They all just seemed like NPC and i couldn't comprehend the idea that they also have thoughts and their own life.I don't know whether I'm just too smart for life or it's the fact that I've been neglected emotionally by parents (idk if its real, found out this week on reddit, but it explains why I've always wanted to leave home).Does anybody relate?"

In [5]:
# Does the decoding work?
print(df["document"].str.contains(r"Ã|â|�", na=False).sum())
#Yes, it does

0


# BERTopic

Create the Vectoriser

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",#Either this or the custom list
    ngram_range=(1, 2),
    min_df=8,
    max_df=0.95,
)
# stop_words=english -> English stop word list
#ngram_range -> allows for 2 words instead of 1 in the vector
#min_df=5 -> A word must appear in at least 8 documents to be included
#max_df=0.5 -> A word can only appear in max 70% of documents, otherwise it will be excluded (only affects topic labels, not clustering, i.e. only affects final keyword representation, not how documents are grouped)

Create the c-TF-IDF transformer

In [8]:
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
#After clustering documents into topics, BERTopic needs to identify the key words for each topic.
#reduce_frequent_words=True -> downweights words that appear in many topics.

In [9]:
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  # already used implicitly
umap_model = UMAP(n_neighbors=13, n_components=5, min_dist=0.0, metric='cosine', random_state=42)#for old dataset: n_neighbors=5, for new: =10; n_neighbors drastically influences number of topics
hdbscan_model = HDBSCAN(min_cluster_size=13, min_samples=4, metric='euclidean', prediction_data=True) #lower min_samples = fewer outliers; min_samples currently 4 (best balance between having many outliers and insensible topics

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Create the BERTopic Model

In [10]:
from bertopic import BERTopic

topic_model = BERTopic(
    #top_n_words=10, #not in old dataset; the more words you put in a topic the less coherent it can become (keep between 10 and 20)
    min_topic_size=20,   # lower = more topics, 20 in original dataset, adjust when needed
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto", # "auto" automatically tries to reduce topic size
    calculate_probabilities=True,
    verbose=True
)

Train the Model

In [11]:
topics, probs = topic_model.fit_transform(docs)

2026-08-30 18:06:46,269 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

2026-08-30 18:07:43,748 - BERTopic - Embedding - Completed ✓
2026-08-30 18:07:43,753 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-30 18:08:05,259 - BERTopic - Dimensionality - Completed ✓
2026-08-30 18:08:05,261 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-30 18:08:05,522 - BERTopic - Cluster - Completed ✓
2026-08-30 18:08:05,528 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-30 18:08:06,044 - BERTopic - Representation - Completed ✓
2026-08-30 18:08:06,046 - BERTopic - Topic reduction - Reducing number of topics
2026-08-30 18:08:06,073 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-30 18:08:06,593 - BERTopic - Representation - Completed ✓
2026-08-30 18:08:06,598 - BERTopic - Topic reduction - Reduced number of topics from 15 to 15


In [6]:
# What does the model look like?
topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,283,-1_dream_reality_low_ocd,Outliers,"[dream, reality, low, ocd, feels like, right, ...",NaN
1,0,87,0_weed_high_started_pretty,Substance Use,"[weed, high, started, pretty, went, school, ag...",NaN
2,1,84,1_like im_sleep_panic_attack,Panic & Health Anxiety,"[like im, sleep, panic, attack, panic attack, ...",NaN
3,2,60,2_therapy_post_finally_enjoy,Therapeutic & Social Support,"[therapy, post, finally, enjoy, friends, schoo...",NaN
4,3,60,3_memory_memories_remember_self,Discussing Symptoms #1,"[memory, memories, remember, self, reality, dr...",NaN
5,4,50,4_tried_supplements_medication_helps,Treatment,"[tried, supplements, medication, helps, taking...",NaN
6,5,46,5_room_study_research_open,Discussing Symptoms #2,"[room, study, research, open, link, awareness,...",NaN
7,6,45,6_pressure_head_breathing_happen,Physical & Sensory Symptoms,"[pressure, head, breathing, happen, weird, sma...",NaN
8,7,40,7_sub_posts_life felt_felt like,Living with DPDR,"[sub, posts, life felt, felt like, know dpdr, ...",NaN
9,8,38,8_blank_human_want_don know,Hopelessness,"[blank, human, want, don know, lost, happy, re...",NaN


Let's inspect the topics further

In [7]:
#topic_model
#topic_info = topic_model.get_topic_info()
#topic_info
#topic_info.head(7)

#Try to show complete 'representation' column
with pd.option_context('display.max_colwidth', None):
    display(topic_model.get_topic_info()[['Topic', 'Count', 'Representation']])

,Topic,Count,Representation
0,-1,283,"[dream, reality, low, ocd, feels like, right, world, dp, scared, dissociation]"
1,0,87,"[weed, high, started, pretty, went, school, ago, panic attack, days, panic]"
2,1,84,"[like im, sleep, panic, attack, panic attack, night, wasnt, fucking, tired, panic attacks]"
3,2,60,"[therapy, post, finally, enjoy, friends, school, lost, lot, family, making]"
4,3,60,"[memory, memories, remember, self, reality, dreams, moment, forget, isnt, wrong]"
5,4,50,"[tried, supplements, medication, helps, taking, effect, thanks, sub, deep, safe]"
6,5,46,"[room, study, research, open, link, awareness, visual, looks, dark, vision]"
7,6,45,"[pressure, head, breathing, happen, weird, small, sleep, related, fog, visual]"
8,7,40,"[sub, posts, life felt, felt like, know dpdr, symptom, guess, disorder, gonna, stuff]"
9,8,38,"[blank, human, want, don know, lost, happy, realize, living, christmas, kind]"


In [12]:
# Look at examples per topic
#df["topic"] = topics #topics is only temporary output, so I need to create a column that is not temporary to inspect the element
df[df["topic"] == 13][["title", "text"]].head(10) #use number from "topic" column (instead of leftmost column)

KeyError: 'Topic'

In [14]:
df = pd.read_csv("dpdr_topics_sortiert.csv", encoding="utf-8", encoding_errors="replace")

In [24]:
# Try different code so that I can read the posts better
for post in df[df["topic"] == 12]["document"].head(15):
    print(post)
    print("\n" + "-"*100 + "\n")

It used to scare me (just a little rant). Having DPDR used to scare me , Id be terrified to move out of bed , I couldnt even look at a family member with out freaking out now its like all that fear anxiety has gone away. But not in a healing way in a way its been pushed down further nothing fazes me anymore. Its like Im stuck in some weird world and healing is going to take a hell of a lot of work time and effort. DPDR /dissociation has saved my life in many ways because I dont no what Id do if I was feeling right now so I thank that part of me but I also want that part to no Im safe now and Im capable of healing

----------------------------------------------------------------------------------------------------

Hopeless Ive had chronic dpdr since 2019 but in 2023 I worked a warehouse job for a week because my dpdr was getting worse, I quit the job felt fine, fast forward two months and out of nowhere my dpdr got worse, Ive been in this worsened state of dpdr for almost two years now

In [13]:
# Inspect single posts further
print(df.loc[181, "document"])

Blank mind mixed with some kind of anxiety I function, Im funny, functional human but when I am alone all kind of bad thoughts get me that my mind become blank. Its like I don't have anything in my mind. Im going to kill myself what have I done with myself. Im 34 and having this, im also lost. I had some problems mostly with panic attacks and anxiety but now blank mind is the biggest problem. I don't know where to go in my life cause i don't have a goal at all. Maybe just to be funny guy and that's it but I don't know who I am. Its like im nobody with a blank mind. That's the worst


Quite a lot of outliers. Can we reduce them?

In [17]:
#new_topics = topic_model.reduce_outliers(docs, topics)
#After runninng this, there was only 1 outlier left. Not good.

In [18]:
# Update the model
#topic_model.update_topics(docs, topics=new_topics)

In [19]:
# What does the new model look like?
#topic_model.get_topic_info()

Save the Model

In [4]:
# Save the model
#topic_model.save("bertopic_model", serialization="safetensors", save_ctfidf=True)
# Load later with:
from bertopic import BERTopic
topic_model = BERTopic.load("bertopic_model")

# WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.

2026-08-30 20:43:23,426 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


Adjust the Number of Topics

In [21]:
# That is a lot of topics. See if I can reduce it
#new_topics = topic_model.reduce_outliers(
#    df["document"].tolist(),
#    topics,
#    strategy="embeddings"
#)

#topic_model.update_topics(df["document"].tolist(), topics=new_topics)


In [22]:
#topic_model = BERTopic.load("bertopic_model", embedding_model="all-MiniLM-L6-v2") # reload the model
#new_topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, strategy="probabilities", threshold=0.05)
#topic_model.update_topics(docs, topics=new_topics)
#topic_model.reduce_topics(docs, nr_topics="auto")

In [23]:
# Let's see what happened
#topic_info = topic_model.get_topic_info()
#print(topic_info)
#The stopword list got ignored help

In [24]:
# Website with cool stuff
# https://maartengr.github.io/BERTopic/api/plotting/topics.html#bertopic.plotting._topics.visualize_topics

Visualise the Data

In [5]:
topic_model.set_topic_labels({-1: "Outliers",
                              0: "Substance Use",
                              1: "Panic & Health Anxiety",
                              2: "Therapeutic & Social Support",
                              3: "Discussing Symptoms #1",
                              4: "Treatment",
                              5: "Discussing Symptoms #2",
                              6: "Physical & Sensory Symptoms",
                              7: "Living with DPDR",
                              8: "Hopelessness",
                              9: "Discussing Symptoms #3",
                              10: "Existential Concerns",
                              11: "Diverse (Causes & Symptom Fluctuation)",
                              12: "Medication Experiences",
                              13: "Diverse (Symptoms & Recovery)"
                              })

In [26]:
# Intertopic distance map
fig = topic_model.visualize_topics(custom_labels=True)

# Show figure in notebook
fig.show()

In [27]:
# Save as html
#fig.write_html("new_bertopic_intertopic_distance_map.html")

Download the Data

In [28]:
# Download data
#df["topic"] = topics
#df["probs"] = probs.max(axis=1)  # save the highest probability per document
#df.to_csv("dpdr_topics.csv", index=False)


In [29]:
# Download the topic_info table
#topic_info = topic_model.get_topic_info()
#topic_info.to_csv("dpdr_topic_overview.csv", index=False)

In [30]:
#For when I want to reload the dataset without rerunning BERTopic
#df = pd.read_csv("dpdr_topics.csv")

In [31]:
print("The code worked, I think")

The code worked, I think


Finished this step!!